In [4]:
!pip install feedparser
!pip install feedparser transformers torch
!pip install python-telegram-bot
!pip install schedule

In [2]:
import feedparser
import html
from transformers import pipeline

# ---------- تنظیمات ----------
MAX_ARTICLES_PER_FEED = 2

RSS_FEEDS = [
    "https://techcrunch.com/feed/",
    "https://www.theverge.com/rss/index.xml",
    "https://www.wired.com/feed/category/gear/latest/rss"
]

# ---------- پاکسازی متن ----------
def clean_text(text):
    return html.unescape(text).encode("utf-8", errors="ignore").decode("utf-8")

# ---------- جمع‌آوری اخبار ----------
def get_news():
    news_items = []
    for feed_url in RSS_FEEDS:
        feed = feedparser.parse(feed_url)
        articles = feed.entries[:MAX_ARTICLES_PER_FEED]
        for a in articles:
            title = clean_text(a.get("title", "No title"))
            link = clean_text(a.get("link", "No link"))
            summary = clean_text(a.get("summary", "No summary"))
            news_items.append({"title": title, "link": link, "summary": summary})
    return news_items

# ---------- خلاصه‌سازی با HuggingFace ----------
def summarize_with_huggingface(news_items):
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
    text = " ".join([n["title"] + ". " + n["summary"] for n in news_items])
    result = summarizer(text, max_length=150, min_length=60, do_sample=False)
    summary = result[0]['summary_text']

    # ساخت پست نهایی
    post = "🚀 Tech Updates Today:\n\n"
    post += summary + "\n\n"
    post += "🔗 Links:\n"
    for n in news_items:
        post += f"- {n['title']}: {n['link']}\n"
    return post

# ---------- ذخیره ----------
def save_post(content):
    print("\n✅ --- متن نهایی پست --- ✅\n")
    print(content)
    with open("post_ready.txt", "w", encoding="utf-8") as f:
        f.write(content)
    print("\n📂 متن آماده در فایل post_ready.txt ذخیره شد.\n")

# ---------- اجرای اصلی ----------
if __name__ == "__main__":
    news = get_news()
    post = summarize_with_huggingface(news)
    save_post(post)

from telegram import Bot

# توکن رباتی که از BotFather گرفتی
TELEGRAM_TOKEN = "8387050274:AAHEnbIeHZtHlM4_4sStadDitMg7ntaSe9k"
# آیدی عددی یا یوزرنیم کانال/چت
CHAT_ID = "@MahAINewsBot"

def send_to_telegram(text):
    bot = Bot(token=TELEGRAM_TOKEN)
    bot.send_message(chat_id=-1002413269656, text=text)

# تست
with open("post_ready.txt", "r", encoding="utf-8") as f:
    content = f.read()

send_to_telegram(content)
print("✅ پست به تلگرام ارسال شد.")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu



✅ --- متن نهایی پست --- ✅

🚀 Tech Updates Today:

Trump says Lachlan and Rupert Murdoch might invest in TikTok deal. VCs are still hiring MBAs, but firms are starting to need other experience more. The foldable iPhone might look like two iPhone Airs stuck together. The touchscreen MacBook Pro rumors are never ending..

🔗 Links:
- VCs are still hiring MBAs, but firms are starting to need other experience more: https://techcrunch.com/2025/09/21/vcs-are-still-hiring-mbas-but-firms-are-starting-to-need-other-experience-more/
- Trump says Lachlan and Rupert Murdoch might invest in TikTok deal: https://techcrunch.com/2025/09/21/white-house-offers-more-details-about-potential-tiktok-deal/
- The foldable iPhone might look like two iPhone Airs stuck together: https://www.theverge.com/news/782366/foldable-details-iphone-air-ifixit-teardown
- The touchscreen MacBook rumors are never ending: https://www.theverge.com/news/782302/touchscreen-oled-macbook-rumors
- 14 Best Fitness Trackers (2025), Te

/tmp/ipython-input-2017615293.py:69: RuntimeWarning: coroutine 'Bot.send_message' was never awaited
  bot.send_message(chat_id=-1002413269656, text=text)


In [3]:
import asyncio
import nest_asyncio
from telegram import Bot

nest_asyncio.apply()

TOKEN = "8387050274:AAHEnbIeHZtHlM4_4sStadDitMg7ntaSe9k"
CHAT_ID = -1002413269656  # chat_id واقعی کانال

async def send_txt_file():
    bot = Bot(token=TOKEN)
    # باز کردن فایل txt و خوندن محتوا
    with open("post_ready.txt", "r", encoding="utf-8") as f:
        content = f.read()
    if content.strip():
        await bot.send_message(chat_id=CHAT_ID, text=content)
    else:
        await bot.send_message(chat_id=CHAT_ID, text="⚠️ فایل post_ready.txt خالیه!")

# اجرا در کولب
await send_txt_file()


In [11]:
import schedule
import time
import nest_asyncio
import asyncio
from telegram import Bot

nest_asyncio.apply()

TOKEN = "8387050274:AAHEnbIeHZtHlM4_4sStadDitMg7ntaSe9k"
CHAT_ID = -1002413269656

async def send_to_telegram(text):
    bot = Bot(token=TOKEN)
    await bot.send_message(chat_id=CHAT_ID, text=text)

def job():
    with open("post_ready.txt", "r", encoding="utf-8") as f:
        content = f.read()
    asyncio.run(send_to_telegram(content))
    print("✅ پست زمان‌بندی‌شده ارسال شد.")

# هر روز ساعت 10:00 صبح
schedule.every().day.at("05:30").do(job)

while True:
    schedule.run_pending()
    time.sleep(1)


✅ پست زمان‌بندی‌شده ارسال شد.


KeyboardInterrupt: 

In [10]:
import datetime
print("🕒 ساعت سیستم:", datetime.datetime.now())

🕒 ساعت سیستم: 2025-09-22 05:28:45.621925
